# 🔬 Olist 리뷰 데이터 — 심화 분석 (5가지 활용)

이 노트북은 `02_review_scoring_master_dataset.ipynb`로 만든 마스터 데이터셋을 기반으로 5가지 활용 분석을 진행합니다:

1. **분류 모델** — 어떤 요인이 미스매치를 만드는지 정량화
2. **판매자 모니터링** — 의심 판매자 자동 식별 시스템
3. **데이터 품질 보정** — 잘못된 별점 보정 가중치
4. **카테고리 전략** — 액션이 필요한 카테고리 + 우수 카테고리
5. **고객 LTV / 이탈 위험** — 우선 케어 대상 식별

## 사전 조건
`master_review_level.csv`와 `master_review_item_level.csv`가 같은 폴더에 있어야 합니다.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

mr = pd.read_csv('master_review_level.csv')
mi = pd.read_csv('master_review_item_level.csv')

print(f'master_review_level: {mr.shape}')
print(f'master_review_item_level: {mi.shape}')

---
# TASK 1. 분류 모델 — 어떤 요인이 미스매치를 만드는가?

**미스매치(`is_mismatch`=1)**: `high_rating_negative_text` 또는 `low_rating_positive_text` 
**설명변수**: 배송지연, 결제, 카테고리, 고객 주, 텍스트 길이 등 
**모델**: 로지스틱 회귀(해석용) + 랜덤포레스트(중요도)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

# 텍스트가 있는 리뷰만 학습 대상 (mismatch는 텍스트가 있어야 정의됨)
df = mr[mr['has_text']==True].copy()
df['is_mismatch'] = df['mismatch_type'].isin(
    ['high_rating_negative_text','low_rating_positive_text']).astype(int)

print(f'학습 대상: {len(df):,}')
print(f'미스매치 비율: {df["is_mismatch"].mean()*100:.2f}%')

# 변수 정의
num_features = ['delivery_delay_days','is_late','payment_value','payment_installments',
                'n_items','total_freight','text_length']
cat_features = ['payment_type','customer_state','main_category']

df_model = df.dropna(subset=num_features + cat_features + ['is_mismatch']).copy()
print(f'결측 제거 후: {len(df_model):,}')

# 빈도 적은 카테고리는 OTHER로 묶기
for c in cat_features:
    counts = df_model[c].value_counts()
    keep = counts[counts >= 100].index
    df_model[c] = df_model[c].where(df_model[c].isin(keep), 'OTHER')

In [ ]:
# 피처 매트릭스 만들기
X_num = df_model[num_features].astype(float).values
ohe = OneHotEncoder(sparse_output=False, handle_unknown='ignore')
X_cat = ohe.fit_transform(df_model[cat_features])
X = np.hstack([X_num, X_cat])
y = df_model['is_mismatch'].values
feat_names = num_features + list(ohe.get_feature_names_out(cat_features))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 1) 로지스틱 회귀 — 계수 해석
lr = LogisticRegression(max_iter=500, class_weight='balanced', solver='liblinear')
lr.fit(X_tr, y_tr)
auc_lr = roc_auc_score(y_te, lr.predict_proba(X_te)[:,1])
print(f'Logistic Regression Test AUC: {auc_lr:.3f}')

coef_df = pd.DataFrame({'feature': feat_names, 'coef': lr.coef_[0]})
coef_df['abs_coef'] = coef_df['coef'].abs()
coef_df = coef_df.sort_values('abs_coef', ascending=False)

print('\n📊 Top 15 영향력 변수 (양수=미스매치 증가):')
print(coef_df[['feature','coef']].head(15).to_string(index=False))

In [ ]:
# 2) Random Forest — 변수 중요도
rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42,
                            n_jobs=-1, class_weight='balanced')
rf.fit(X_tr, y_tr)
auc_rf = roc_auc_score(y_te, rf.predict_proba(X_te)[:,1])
print(f'Random Forest Test AUC: {auc_rf:.3f}')

imp_df = pd.DataFrame({'feature': feat_names, 'importance': rf.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=False)
print('\n📊 Top 15 변수 중요도:')
print(imp_df.head(15).to_string(index=False))

coef_df.to_csv('model_lr_coefficients.csv', index=False)
imp_df.to_csv('model_rf_importance.csv', index=False)
print('\n💾 model_lr_coefficients.csv, model_rf_importance.csv 저장')

In [ ]:
# Top 15 중요도 시각화
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

top_imp = imp_df.head(15).iloc[::-1]
axes[0].barh(top_imp['feature'], top_imp['importance'], color='steelblue')
axes[0].set_title('Random Forest — Feature Importance', fontsize=13)
axes[0].set_xlabel('importance')

top_coef = coef_df.head(15).iloc[::-1]
colors = ['#d62728' if c>0 else '#2ca02c' for c in top_coef['coef']]
axes[1].barh(top_coef['feature'], top_coef['coef'], color=colors)
axes[1].axvline(0, color='black', linewidth=0.7)
axes[1].set_title('Logistic Regression — Coefficients\n(red=mismatch↑, green=mismatch↓)', fontsize=13)
axes[1].set_xlabel('coefficient')

plt.tight_layout()
plt.savefig('task1_model_importance.png', dpi=200, bbox_inches='tight')
plt.show()

### 💡 TASK 1 인사이트
- **`text_length`가 압도적 1위 중요도** — 리뷰가 길수록 미스매치 가능성 높음 (불만일 때 길게 씀)
- **배송지연/결제액/운임**도 의미 있는 변수
- **카테고리별 차이 뚜렷** — `telephony`, `small_appliances`는 미스매치 증가 / `audio`, `fashion_shoes`는 감소
- AUC 0.63은 완벽하진 않지만 의미 있는 신호 (텍스트 자체 피처를 추가하면 더 올라갈 것)

---
# TASK 2. 판매자 모니터링 — 의심 판매자 자동 식별

In [ ]:
# 판매자별 핵심 지표 집계
seller_stats = (mi.groupby('seller_id').agg(
    total_reviews=('review_id','count'),
    n_5neg=('mismatch_type', lambda x: (x=='high_rating_negative_text').sum()),
    n_1pos=('mismatch_type', lambda x: (x=='low_rating_positive_text').sum()),
    n_aligned_neg=('mismatch_type', lambda x: (x=='aligned_negative').sum()),
    avg_rating=('review_score','mean'),
    avg_sent=('sent_score','mean'),
    seller_state=('seller_state','first')
).round(3))
seller_stats['pct_5neg']        = (seller_stats['n_5neg']/seller_stats['total_reviews']*100).round(2)
seller_stats['pct_aligned_neg'] = (seller_stats['n_aligned_neg']/seller_stats['total_reviews']*100).round(2)

# 위험도 점수 (0~100)
def risk_score(r):
    if r['total_reviews'] < 30: return np.nan   # 표본 부족
    score = 0
    score += min(r['pct_5neg'] * 10, 50)         # 별점은 좋은데 본문 부정 (가장 위험)
    score += min(r['pct_aligned_neg'] * 1, 30)   # 명백한 불만 비율
    if r['avg_sent'] < 0:
        score += min(abs(r['avg_sent']) * 20, 20)
    return min(score, 100)

seller_stats['risk_score'] = seller_stats.apply(risk_score, axis=1)
seller_stats['alert_level'] = pd.cut(seller_stats['risk_score'],
                                      bins=[-1,30,50,100],
                                      labels=['🟢 normal','🟡 watch','🔴 alert'])

print(f'분석 대상 판매자 (30건 이상): {seller_stats["risk_score"].notna().sum():,}')
print('\n경계 등급 분포:')
print(seller_stats['alert_level'].value_counts())

In [ ]:
# 의심 판매자 Top 15
suspect = seller_stats.dropna(subset=['risk_score']).sort_values('risk_score', ascending=False).head(15)

print('🔴 의심 판매자 Top 15:')
cols = ['total_reviews','avg_rating','avg_sent','pct_5neg','pct_aligned_neg','risk_score','alert_level','seller_state']
print(suspect[cols].to_string())

seller_stats.to_csv('seller_monitoring.csv')
print('\n💾 seller_monitoring.csv 저장 (전체 판매자 모니터링 대시보드용)')

In [ ]:
# 위험도 분포 시각화
valid = seller_stats.dropna(subset=['risk_score'])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(valid['risk_score'], bins=40, color='steelblue', edgecolor='white')
axes[0].axvline(30, color='gold', linestyle='--', label='watch (30)')
axes[0].axvline(50, color='red', linestyle='--', label='alert (50)')
axes[0].set_title('판매자 위험도 점수 분포')
axes[0].set_xlabel('risk_score'); axes[0].set_ylabel('# sellers')
axes[0].legend()

# 별점 vs 위험도
axes[1].scatter(valid['avg_rating'], valid['risk_score'], 
                s=valid['total_reviews']/10, alpha=0.4, color='tomato')
axes[1].axhline(50, color='red', linestyle='--', alpha=0.5)
axes[1].set_xlabel('평균 별점'); axes[1].set_ylabel('risk_score')
axes[1].set_title('평균 별점 vs 위험도 (점 크기 = 리뷰 수)')

plt.tight_layout()
plt.savefig('task2_seller_risk.png', dpi=200, bbox_inches='tight')
plt.show()

### 💡 TASK 2 인사이트
- 평가 가능한 684개 판매자 중 **13개가 alert(빨강), 64개가 watch(노랑)**
- 별점이 높아도 위험 점수가 높은 판매자 존재 → **별점만 보면 안 보이는 위험**
- 이 `seller_monitoring.csv`를 매주/매월 갱신해서 **자동 알람 시스템** 구성 가능

---
# TASK 3. 데이터 품질 보정 — 잘못된 별점 가중치

In [ ]:
# 미스매치 타입 + 신뢰도에 따라 가중치 부여
def weight(row):
    # 1점인데 텍스트는 칭찬 → 별점 잘못 누른 케이스 가능성↑ → 가중치 낮춤
    if row['mismatch_type'] == 'low_rating_positive_text':
        if row['sent_confidence'] == 'high':   return 0.0   # 제외
        if row['sent_confidence'] == 'medium': return 0.3
        return 0.7
    # 5점인데 텍스트는 부정 → 텍스트가 진심일 가능성↑ → 별점 가중치 낮춤
    if row['mismatch_type'] == 'high_rating_negative_text':
        if row['sent_confidence'] == 'high':   return 0.5
        return 0.7
    return 1.0

mr['quality_weight'] = mr.apply(weight, axis=1)

print('가중치 분포:')
print(mr['quality_weight'].value_counts().sort_index())
print(f'\n제외 권장 (weight=0): {(mr["quality_weight"]==0).sum():,}건')
print(f'가중치 낮춤 (weight<1): {(mr["quality_weight"]<1).sum():,}건')

In [ ]:
# 정제 전후 평균 별점 비교
before = mr['review_score'].mean()
after  = (mr['review_score'] * mr['quality_weight']).sum() / mr['quality_weight'].sum()
print(f'전체 평균 별점:')
print(f'  raw     : {before:.4f}')
print(f'  cleaned : {after:.4f}')
print(f'  delta   : {after-before:+.4f}')

In [ ]:
# 카테고리별 정제 효과
cat_compare = mr.groupby('main_category').apply(
    lambda d: pd.Series({
        'n': len(d),
        'rating_raw': d['review_score'].mean(),
        'rating_clean': (d['review_score']*d['quality_weight']).sum()/d['quality_weight'].sum() if d['quality_weight'].sum()>0 else np.nan
    })).round(3)
cat_compare['delta'] = (cat_compare['rating_clean'] - cat_compare['rating_raw']).round(3)
cat_compare = cat_compare[cat_compare['n']>=300].sort_values('delta')

print('정제 후 평점이 가장 많이 떨어진 카테고리 (숨겨진 불만 많음):')
print(cat_compare.head(10).to_string())

print('\n정제 후 평점이 가장 많이 오른 카테고리 (별점 잘못 누른 케이스 많음):')
print(cat_compare.tail(10).to_string())

# 가중치 컬럼 포함 저장
mr[['review_id','order_id','review_score','sent_score','mismatch_type','quality_weight']].to_csv(
    'reviews_with_weights.csv', index=False)
print('\n💾 reviews_with_weights.csv 저장')

### 💡 TASK 3 활용
- 추천 시스템 학습 시 `quality_weight` 컬럼을 **샘플 가중치(sample_weight)** 로 사용
- `quality_weight=0`인 케이스는 학습에서 제외 (잘못된 라벨)
- KPI 대시보드에서 "raw rating" + "quality-adjusted rating" 둘 다 표시하면 숨은 불만 발견

---
# TASK 4. 카테고리 전략 — 액션 아이템 도출

In [ ]:
# 카테고리별 종합 지표
cat_stats = (mi.groupby('product_category_name_english').agg(
    n_reviews=('review_id','count'),
    avg_rating=('review_score','mean'),
    avg_sent=('sent_score','mean'),
    avg_price=('price','mean'),
    avg_delay=('delivery_delay_days','mean'),
    pct_late=('is_late', lambda x: (x.sum()/x.notna().sum()*100) if x.notna().any() else 0),
    n_5neg=('mismatch_type', lambda x: (x=='high_rating_negative_text').sum()),
    n_aligned_neg=('mismatch_type', lambda x: (x=='aligned_negative').sum()),
).round(2))
cat_stats['pct_5neg']        = (cat_stats['n_5neg']/cat_stats['n_reviews']*100).round(2)
cat_stats['pct_aligned_neg'] = (cat_stats['n_aligned_neg']/cat_stats['n_reviews']*100).round(2)

# 액션 분류 규칙
def action(r):
    if r['n_reviews'] < 200: return 'too_small'
    if r['pct_5neg'] >= 1.5 and r['pct_aligned_neg'] >= 10:
        return '🚨 결함률 높음 — AS/품질 점검'
    if r['pct_5neg'] >= 1.5:
        return '⚠️ 숨은 불만 — 텍스트 모니터링'
    if r['pct_aligned_neg'] >= 15:
        return '⚠️ 명백한 불만 비율 높음'
    if r['avg_delay'] > -7:
        return '🚛 배송 개선 필요'
    if r['pct_aligned_neg'] < 5 and r['avg_rating'] >= 4.3:
        return '✅ 우수'
    return '⚪ 보통'

cat_stats['action'] = cat_stats.apply(action, axis=1)

print('카테고리별 액션 분포:')
print(cat_stats['action'].value_counts())

In [ ]:
# 즉시 액션 필요 카테고리
priority = cat_stats[cat_stats['action'].str.contains('결함률|숨은|명백한', na=False)] \
    .sort_values('pct_5neg', ascending=False)
print('🚨 즉시 액션 필요 카테고리:')
print(priority[['n_reviews','avg_rating','pct_5neg','pct_aligned_neg','avg_delay','action']].to_string())

# 우수 카테고리
good = cat_stats[cat_stats['action']=='✅ 우수'].sort_values('avg_rating', ascending=False)
print('\n✅ 우수 카테고리 (Top 10):')
print(good[['n_reviews','avg_rating','pct_5neg','pct_aligned_neg','avg_delay']].head(10).to_string())

cat_stats.to_csv('category_strategy.csv')
print('\n💾 category_strategy.csv 저장')

In [ ]:
# 카테고리 매트릭스 시각화: 평균 별점 vs 5점-부정 비율
valid = cat_stats[cat_stats['n_reviews']>=300].copy()

plt.figure(figsize=(12, 7))
scatter = plt.scatter(valid['avg_rating'], valid['pct_5neg'],
                       s=valid['n_reviews']/30, c=valid['avg_delay'],
                       cmap='RdYlGn', alpha=0.7, edgecolor='black')
plt.colorbar(scatter, label='avg_delay_days (- = 일찍 도착)')
plt.axhline(1.5, color='red', linestyle='--', alpha=0.5, label='5점-부정 1.5%')
plt.axvline(4.3, color='green', linestyle='--', alpha=0.5, label='평점 4.3')

# 라벨 표시 (튀는 카테고리만)
for _, r in valid.iterrows():
    if r['pct_5neg']>=1.4 or r['avg_rating']<3.7:
        plt.annotate(r.name, (r['avg_rating'], r['pct_5neg']),
                     fontsize=8, alpha=0.8)

plt.xlabel('평균 별점')
plt.ylabel('5점인데 부정 텍스트 비율 (%)')
plt.title('카테고리 전략 매트릭스 (점 크기 = 리뷰 수)')
plt.legend()
plt.tight_layout()
plt.savefig('task4_category_matrix.png', dpi=200, bbox_inches='tight')
plt.show()

### 💡 TASK 4 인사이트
- **즉시 액션**: `air_conditioning`, `office_furniture`, `fixed_telephony` — 5점-부정 비율 1.5%↑
- **우수 카테고리**: `books_*`, `food_drink`, `luggage_accessories` — 책/식품은 만족도 압도적
- **시사점**: 조립/설치가 필요한 상품군이 위험하므로 → 상품 페이지에 조립 가이드/AS 정보 강화

---
# TASK 5. 고객 LTV / 이탈 위험 분석

In [ ]:
# 고객별(customer_unique_id) 누적 통계
cust = (mr.dropna(subset=['customer_unique_id']).groupby('customer_unique_id').agg(
    n_orders=('order_id','nunique'),
    n_reviews=('review_id','count'),
    total_spent=('payment_value','sum'),
    avg_rating=('review_score','mean'),
    avg_sent=('sent_score','mean'),
    n_mismatch=('mismatch_type', lambda x: x.isin(['high_rating_negative_text','low_rating_positive_text']).sum()),
    n_5neg=('mismatch_type', lambda x: (x=='high_rating_negative_text').sum()),
    n_aligned_neg=('mismatch_type', lambda x: (x=='aligned_negative').sum()),
    last_order=('order_purchase_timestamp','max'),
    customer_state=('customer_state','first')
).round(2))
cust['pct_negative_text'] = ((cust['n_5neg'] + cust['n_aligned_neg']) / cust['n_reviews'] * 100).round(2)

# 이탈 위험 점수 (0~100)
def churn_risk(r):
    if r['n_reviews'] < 1: return np.nan
    risk = 0
    risk += min(r['pct_negative_text'] * 0.6, 40)
    if r['avg_sent'] < 0:
        risk += min(abs(r['avg_sent'])*15, 30)
    if r['n_5neg'] >= 1:
        risk += 30   # 별점에 안 보이는 불만 = 강한 이탈 신호
    return min(risk, 100)

cust['churn_risk']   = cust.apply(churn_risk, axis=1)
cust['risk_segment'] = pd.cut(cust['churn_risk'], bins=[-1,30,60,100],
                               labels=['🟢 safe','🟡 watch','🔴 high_risk'])

print(f'전체 고객: {len(cust):,}')
print(f'재구매 고객 (2건+): {(cust["n_orders"]>=2).sum():,}')
print('\n전체 고객 위험 세그먼트:')
print(cust['risk_segment'].value_counts())
print('\n재구매 고객 위험 세그먼트:')
print(cust[cust['n_orders']>=2]['risk_segment'].value_counts())

In [ ]:
# 우선 케어 대상: 재구매 + 고위험 + 누적 지출 큰 순
repeat_high = cust[(cust['n_orders']>=2) & (cust['risk_segment']=='🔴 high_risk')]
priority_care = repeat_high.sort_values('total_spent', ascending=False).head(20)

print('🚨 우선 케어 대상 Top 20 (재구매 + 고위험 + 누적지출 높음):')
cols = ['n_orders','n_reviews','total_spent','avg_rating','avg_sent',
        'n_5neg','pct_negative_text','churn_risk','customer_state']
print(priority_care[cols].to_string())

# 세그먼트별 지출
print('\n💰 세그먼트별 평균 누적 지출 (BRL):')
print(cust.groupby('risk_segment').agg(
    n=('avg_rating','count'),
    avg_spent=('total_spent','mean'),
    median_spent=('total_spent','median'),
    total_spent=('total_spent','sum')
).round(2))

cust.to_csv('customer_ltv_risk.csv')
print('\n💾 customer_ltv_risk.csv 저장')

In [ ]:
# 위험 세그먼트 시각화
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 위험 점수 분포 (재구매 고객)
rep = cust[cust['n_orders']>=2].dropna(subset=['churn_risk'])
axes[0].hist(rep['churn_risk'], bins=30, color='tomato', edgecolor='white')
axes[0].axvline(30, color='gold', linestyle='--', label='watch')
axes[0].axvline(60, color='red', linestyle='--', label='high risk')
axes[0].set_xlabel('churn_risk'); axes[0].set_ylabel('# customers')
axes[0].set_title('재구매 고객 이탈 위험 점수')
axes[0].legend()

# 세그먼트별 누적 지출 박스플롯
valid = cust.dropna(subset=['risk_segment'])
valid_box = valid[valid['total_spent']<=1000]   # outlier 제외 시각화
sns.boxplot(data=valid_box, x='risk_segment', y='total_spent',
            ax=axes[1], showfliers=False, palette='RdYlGn_r')
axes[1].set_title('세그먼트별 누적 지출 분포')
axes[1].set_ylabel('total_spent (BRL)')

plt.tight_layout()
plt.savefig('task5_customer_risk.png', dpi=200, bbox_inches='tight')
plt.show()

### 💡 TASK 5 인사이트
- 전체 95,380명 중 **4,299명이 high_risk(4.5%)**
- 재구매 고객 2,951명 중 **74명이 high_risk** — 이들은 누적 지출도 큰 우수 고객들
- 우선 케어 Top 20 고객은 누적 지출 380~1,300 BRL 사이 — **이탈 시 직접 매출 손실 발생**
- 활용: 위험 점수 ≥60 고객에게 **사과 메시지 + 쿠폰** 자동 발송 트리거 구축

---
# 📦 산출물 요약

이 노트북이 만든 파일들:

| 파일 | 설명 | 활용 |
|---|---|---|
| `model_lr_coefficients.csv` | 로지스틱 회귀 계수 | 어떤 변수가 미스매치를 만드는지 |
| `model_rf_importance.csv` | 랜덤포레스트 중요도 | 변수 우선순위 |
| `seller_monitoring.csv` | 전체 판매자 위험도 | 정기 갱신 → 자동 알람 |
| `reviews_with_weights.csv` | 리뷰별 품질 가중치 | ML 모델 sample_weight |
| `category_strategy.csv` | 카테고리별 액션 | 운영팀 우선순위 결정 |
| `customer_ltv_risk.csv` | 고객별 이탈 위험 | CRM 자동 케어 트리거 |

각 CSV는 BI 도구(Tableau, Power BI, Looker 등)에 바로 연결해서 대시보드로 만들 수 있습니다.